In [ ]:
"""
=============================================================
FILE 39 — BLACKBOARD ARCHITECTURE
=============================================================

CONCEPTS TAUGHT
----------------
1. Blackboard Architecture
2. Shared State Collaboration
3. Multi-Agent Coordination
4. Shared Memory Systems
5. Collaborative Intelligence
6. Distributed Problem Solving
7. Agent Communication
8. Cooperative Reasoning
9. Shared Knowledge Base
10. Enterprise Collaboration Systems

CORE IDEA
-----------
Multiple agents collaborate
through a shared blackboard/state.

FLOW
-----
Shared Blackboard
   ↓
Agent Reads Blackboard
   ↓
Agent Updates Blackboard
   ↓
Other Agents Continue

REAL WORLD USE CASES
---------------------
- Enterprise collaboration
- AI research systems
- Team-based reasoning
- Collaborative planning
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — BLACKBOARD STATE
# ============================================================

class State(TypedDict):

    problem: str

    blackboard: str

    finance_notes: str
    marketing_notes: str
    strategy_notes: str

# ============================================================
# STEP 5 — AGENT 1
# ============================================================

def finance_agent(state: State):

    response = llm.invoke(
        f"""
        Analyze this business problem
        from finance perspective.

        PROBLEM:
        {state['problem']}
        """
    )

    return {
        "finance_notes": response.content,
        "blackboard":
        f"""
        Finance Analysis:
        {response.content}
        """
    }

# ============================================================
# STEP 6 — AGENT 2
# ============================================================

def marketing_agent(state: State):

    response = llm.invoke(
        f"""
        Read the shared blackboard below.

        BLACKBOARD:
        {state['blackboard']}

        Add marketing analysis.
        """
    )

    return {
        "marketing_notes": response.content,
        "blackboard":
        state["blackboard"]
        +
        f"""

        Marketing Analysis:
        {response.content}
        """
    }

# ============================================================
# STEP 7 — AGENT 3
# ============================================================

def strategy_agent(state: State):

    response = llm.invoke(
        f"""
        Read the entire blackboard.

        BLACKBOARD:
        {state['blackboard']}

        Create final enterprise strategy.
        """
    )

    return {
        "strategy_notes": response.content
    }

# ============================================================
# STEP 8 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("finance_agent", finance_agent)
builder.add_node("marketing_agent", marketing_agent)
builder.add_node("strategy_agent", strategy_agent)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

builder.add_edge(START, "finance_agent")

builder.add_edge(
    "finance_agent",
    "marketing_agent"
)

builder.add_edge(
    "marketing_agent",
    "strategy_agent"
)

builder.add_edge(
    "strategy_agent",
    END
)

# ============================================================
# STEP 10 — COMPILE
# ============================================================

graph = builder.compile()

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 11 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "problem":
        """
        AI adoption roadmap for a retail bank
        """
    }
)

# ============================================================
# STEP 12 — PRINT RESULT
# ============================================================

print("\nFINAL STRATEGY\n")
print("=" * 60)

print(result["strategy_notes"])